## Model Inference

## 0. Setup

In [ ]:
!pip install -q neuralforecast wandb prophet lightgbm xgboost

In [ ]:
import glob, os, time, pickle
import numpy as np
import pandas as pd

import wandb

WANDB_PROJECT = "walmart-sales-forecasting"
wandb.login()

_search_roots = ["/kaggle/input", "data/raw", "data", "."]
DATA_DIR = None
for _root in _search_roots:
    _matches = glob.glob(f"{_root}/**/train.csv*", recursive=True)
    if _matches:
        DATA_DIR = os.path.dirname(_matches[0])
        break

if DATA_DIR is None:
    if os.path.isdir("/kaggle/input"):
        raise FileNotFoundError(
            "train.csv ვერ მოიძებნა /kaggle/input-ში. მიაბი კონკურსის მონაცემები:\n"
            "  Add Input -> Competitions -> 'Walmart Recruiting - Store Sales Forecasting'\n"
            "(კონკურსის გვერდზე წესებზე დათანხმება წინაპირობაა), შემდეგ თავიდან გაუშვი ეს cell-ი."
        )
    raise FileNotFoundError(
        "train.csv ვერ მოიძებნა. ლოკალურად notebook-ი repo root-იდან გაუშვი, "
        "სადაც data/raw/train.csv დევს, ან ჩამოტვირთე კონკურსის მონაცემები data/raw-ში."
    )
print("DATA_DIR ->", DATA_DIR)

HORIZON = 39

## 1. Leaderboard — ყველა მოდელის საბოლოო WMAE და best config wandb-დან

თითო მოდელზე ვიღებთ უახლეს `job_type='final'` run-ს, რომელსაც შესაბამისი მეტრიკა
აქვს დალოგილი. `run.config` სწორედ ის best config-ია, რომლითაც საბოლოო მოდელი
გაწვრთნა (ყველა ტრენინგ-notebook `final` run-ს `config=best_config`-ით ხსნის).

In [ ]:
UNIFIED_METRIC = "val_WMAE"
MODEL_METRICS = {m: UNIFIED_METRIC for m in
                 ["ARIMA", "SARIMA", "NBEATS", "TFT",
                  "Prophet", "LightGBM", "XGBoost", "DLinear"]}
REGISTRY_MODELS = {"ARIMA", "SARIMA", "NBEATS", "TFT"}

api = wandb.Api()
entity = api.default_entity
rows = []
for model_name, metric in MODEL_METRICS.items():
    try:
        runs = api.runs(f"{entity}/{WANDB_PROJECT}",
                        filters={"group": model_name, "jobType": "final"},
                        order="-created_at")
        latest = next((r for r in runs if r.summary.get(metric) is not None), None)
    except Exception as e:
        print(f"  {model_name}: wandb query ჩავარდა ({e}) — გამოტოვებულია.")
        continue
    if latest is None:
        print(f"  {model_name}: final run მეტრიკით '{metric}' ვერ მოიძებნა — "
              f"გამოტოვებულია (ალბათ notebook ჯერ არ გადაშვებულა 39w-holdout ვერსიით).")
        continue
    best_config = {k: v for k, v in latest.config.items() if not k.startswith("_")}
    rows.append({"model": model_name, "wmae": float(latest.summary[metric]),
                 "metric": metric, "final_run": latest.name,
                 "created_at": latest.created_at, "best_config": best_config})

leaderboard = pd.DataFrame(rows).sort_values("wmae").reset_index(drop=True)
assert len(leaderboard) > 0, "ვერცერთი მოდელის final run ვერ მოიძებნა wandb-ში!"

BEST_MODEL = str(leaderboard.iloc[0]["model"])
BEST_WMAE = float(leaderboard.iloc[0]["wmae"])
BEST_CONFIG = dict(leaderboard.iloc[0]["best_config"])

print("\n=== Leaderboard (val_WMAE, 39-კვირიანი holdout — ერთიანი პროტოკოლი) ===")
print(leaderboard.drop(columns=["best_config"]).to_string(index=False))
print(f"\nგამარჯვებული: {BEST_MODEL} (wmae={BEST_WMAE:.2f}, metric={leaderboard.iloc[0]['metric']})")
print("best config:", BEST_CONFIG)

## 2. საერთო მონაცემები და helper-ები

Raw CSV-ები + იგივე გასუფთავება/feature engineering, რაც ტრენინგ-notebook-ებში.
ეს სჭირდება (ა) lookup/NF pipeline-ების fallback-საშუალოებს და (ბ) `unique_id`/`ds`
სვეტების აგებას test-ისთვის.

In [ ]:
train_raw = pd.read_csv(glob.glob(f"{DATA_DIR}/train.csv*")[0], parse_dates=["Date"])
test_raw = pd.read_csv(glob.glob(f"{DATA_DIR}/test.csv*")[0], parse_dates=["Date"])
stores = pd.read_csv(glob.glob(f"{DATA_DIR}/stores.csv*")[0])
features = pd.read_csv(glob.glob(f"{DATA_DIR}/features.csv*")[0], parse_dates=["Date"])

def merge_sources(df):
    feat = features.drop(columns=["IsHoliday"])
    return df.merge(stores, on="Store", how="left").merge(feat, on=["Store", "Date"], how="left")

def clean_data(df, is_train=True):
    df = df.copy()
    df = df.drop_duplicates()

    markdown_cols = [c for c in df.columns if c.startswith("MarkDown")]
    df[markdown_cols] = df[markdown_cols].fillna(0).clip(lower=0)

    df = df.sort_values(["Store", "Date"])
    for col in ["CPI", "Unemployment"]:
        df[col] = df.groupby("Store")[col].ffill().bfill()

    if is_train:
        df["Weekly_Sales"] = df["Weekly_Sales"].clip(lower=0)

    df["IsHoliday"] = df["IsHoliday"].astype(bool)
    df["Store"] = df["Store"].astype(int)
    df["Dept"] = df["Dept"].astype(int)
    return df

def add_calendar_flags(df):
    df = df.copy()
    df["Year"] = df["Date"].dt.year
    df["Month"] = df["Date"].dt.month
    df["WeekOfYear"] = df["Date"].dt.isocalendar().week.astype(int)

    superbowl = pd.to_datetime(["2010-02-12", "2011-02-11", "2012-02-10", "2013-02-08"])
    labor_day = pd.to_datetime(["2010-09-10", "2011-09-09", "2012-09-07", "2013-09-06"])
    thanksgiving = pd.to_datetime(["2010-11-26", "2011-11-25", "2012-11-23", "2013-11-29"])
    christmas = pd.to_datetime(["2010-12-31", "2011-12-30", "2012-12-28", "2013-12-27"])
    df["IsSuperBowl"] = df["Date"].isin(superbowl)
    df["IsLaborDay"] = df["Date"].isin(labor_day)
    df["IsThanksgiving"] = df["Date"].isin(thanksgiving)
    df["IsChristmas"] = df["Date"].isin(christmas)
    return df

def engineer_features(df):
    df = add_calendar_flags(df)
    df["unique_id"] = df["Store"].astype(str) + "_" + df["Dept"].astype(str)
    df["ds"] = df["Date"]
    return df

train_clean = clean_data(merge_sources(train_raw), is_train=True)
train_fe = engineer_features(train_clean)
train_fe["y"] = train_fe["Weekly_Sales"]
print("history rows:", len(train_fe), "| series:", train_fe["unique_id"].nunique())

class FallbackMeans:
    def __init__(self, history_df):
        self._dept_mean = history_df.groupby(["Store", "Dept"])["y"].mean()
        self._store_mean = history_df.groupby("Store")["y"].mean()
        self._global_mean = float(history_df["y"].mean())

    def value(self, store, dept):
        key = (store, dept)
        if key in self._dept_mean.index:
            return float(self._dept_mean[key])
        if store in self._store_mean.index:
            return float(self._store_mean[store])
        return self._global_mean

fallback = FallbackMeans(train_fe)

## 3. Family-loaders

In [ ]:
class LookupForecastPipeline:

    def __init__(self, payload, fallback):
        self.forecasts = payload["forecasts"]
        self.last_date = payload["last_date"]
        self.fallback = fallback

    def predict(self, raw_df):
        df = raw_df.copy()
        df["Date"] = pd.to_datetime(df["Date"])
        df = engineer_features(clean_data(merge_sources(df), is_train=False))

        out_rows = []
        for uid, g in df.groupby("unique_id"):
            store, dept = int(g["Store"].iloc[0]), int(g["Dept"].iloc[0])
            dates = g.sort_values("ds")["ds"].tolist()
            fc_vals = self.forecasts.get(uid)

            preds = [np.nan] * len(dates)
            if fc_vals is not None:
                future_dates = pd.date_range(self.last_date + pd.Timedelta(weeks=1),
                                             periods=len(fc_vals), freq="W-FRI")
                fc_map = dict(zip(future_dates, fc_vals))
                preds = [fc_map.get(d, np.nan) for d in dates]

            out_rows.append(pd.DataFrame({"unique_id": uid, "ds": dates, "Store": store,
                                           "Dept": dept, "Weekly_Sales_Pred": preds}))

        out = pd.concat(out_rows, ignore_index=True)
        missing = out["Weekly_Sales_Pred"].isna()
        if missing.any():
            out.loc[missing, "Weekly_Sales_Pred"] = out.loc[missing].apply(
                lambda r: self.fallback.value(r["Store"], r["Dept"]), axis=1)
            print(f"LookupForecastPipeline: filled {int(missing.sum())}/{len(out)} rows via fallback mean.")
        return out.drop(columns=["Store", "Dept"])

def load_lookup(pipeline_dir):
    pkl = glob.glob(f"{pipeline_dir}/**/fitted_models.pkl", recursive=True)[0]
    with open(pkl, "rb") as f:
        payload = pickle.load(f)
    print("lookup pipeline:", {k: v for k, v in payload.items() if k != "forecasts"},
          "| n_series:", len(payload["forecasts"]))
    return LookupForecastPipeline(payload, fallback)

In [ ]:
class NFPipeline:

    def __init__(self, nf, fallback, futr_builder=None):
        self.nf = nf
        self.fallback = fallback
        self.futr_builder = futr_builder

    def predict(self, raw_df):
        df = raw_df.copy()
        df["Date"] = pd.to_datetime(df["Date"])
        df = engineer_features(clean_data(merge_sources(df), is_train=False))

        if self.futr_builder is not None:
            preds = self.nf.predict(futr_df=self.futr_builder(self.nf))
        else:
            preds = self.nf.predict()
        model_col = [c for c in preds.columns if c not in ("unique_id", "ds")][0]
        out = df[["unique_id", "ds", "Store", "Dept"]].merge(preds, on=["unique_id", "ds"], how="left")
        out = out.rename(columns={model_col: "Weekly_Sales_Pred"})

        missing = out["Weekly_Sales_Pred"].isna()
        if missing.any():
            out.loc[missing, "Weekly_Sales_Pred"] = out.loc[missing].apply(
                lambda r: self.fallback.value(r["Store"], r["Dept"]), axis=1)
            print(f"NFPipeline: filled {int(missing.sum())}/{len(out)} rows via fallback mean.")
        return out.drop(columns=["Store", "Dept"])

def _nf_load(pipeline_dir):
    from neuralforecast import NeuralForecast
    nf_dir = os.path.join(pipeline_dir, "nf_model")
    if not os.path.isdir(nf_dir):
        nf_dir = pipeline_dir
    nf = NeuralForecast.load(path=nf_dir)
    print("ჩატვირთული NF მოდელ(ებ)ი:", [type(m).__name__ for m in nf.models])
    return nf

def load_nf(pipeline_dir):
    return NFPipeline(_nf_load(pipeline_dir), fallback)

def load_tft(pipeline_dir):
    nf = _nf_load(pipeline_dir)
    futr_cols = list(getattr(nf.models[0], "futr_exog_list", []) or [])
    print("TFT futr_exog_list (მოდელიდანვე):", futr_cols)

    store_exog = features.merge(stores, on="Store", how="left")
    md_cols = [c for c in store_exog.columns if c.startswith("MarkDown")]
    store_exog[md_cols] = store_exog[md_cols].fillna(0).clip(lower=0)
    store_exog = store_exog.sort_values(["Store", "Date"])
    for col in ["CPI", "Unemployment"]:
        store_exog[col] = store_exog.groupby("Store")[col].ffill().bfill()
    store_exog["Store"] = store_exog["Store"].astype(int)
    store_exog = add_calendar_flags(store_exog).rename(columns={"Date": "ds"})
    for c in store_exog.columns:
        if store_exog[c].dtype == bool:
            store_exog[c] = store_exog[c].astype(int)

    def build_futr(nf_obj):
        futr = nf_obj.make_future_dataframe()
        futr["Store"] = futr["unique_id"].str.split("_").str[0].astype(int)
        futr = futr.merge(store_exog[["Store", "ds"] + futr_cols], on=["Store", "ds"], how="left")
        futr = futr.drop(columns=["Store"])
        futr = futr.groupby("unique_id", group_keys=False).apply(lambda g: g.ffill().bfill())
        return futr

    return NFPipeline(nf, fallback, futr_builder=build_futr)

In [ ]:
from sklearn.base import BaseEstimator, RegressorMixin

MARKDOWN_COLS = [f"MarkDown{i}" for i in range(1, 6)]

def add_calendar_features(df):
    df = df.copy()
    df["Year"] = df["Date"].dt.year
    df["Month"] = df["Date"].dt.month
    df["Week"] = df["Date"].dt.isocalendar().week.astype(int)
    df["DayOfYear"] = df["Date"].dt.dayofyear
    df["Quarter"] = df["Date"].dt.quarter
    df["Month_sin"] = np.sin(2 * np.pi * df["Month"] / 12)
    df["Month_cos"] = np.cos(2 * np.pi * df["Month"] / 12)
    df["Week_sin"] = np.sin(2 * np.pi * df["Week"] / 52)
    df["Week_cos"] = np.cos(2 * np.pi * df["Week"] / 52)
    return df

def merge_static(df, stores_df, features_df):
    df = df.merge(stores_df, on="Store", how="left")
    feats = features_df.drop(columns=["IsHoliday"])
    df = df.merge(feats, on=["Store", "Date"], how="left")
    for c in MARKDOWN_COLS:
        df[c] = df[c].fillna(0.0)
    for c in ("Temperature", "Fuel_Price", "CPI", "Unemployment"):
        df[c] = df[c].fillna(df[c].median())
    return df

def target_encode_fit(X, y):
    y = pd.Series(np.asarray(y, dtype=float), index=X.index)
    store_mean = y.groupby(X["Store"]).mean()
    dept_mean = y.groupby(X["Dept"]).mean()
    storedept_mean = y.groupby([X["Store"], X["Dept"]]).mean()
    global_mean = float(y.mean())
    return {"store": store_mean, "dept": dept_mean, "storedept": storedept_mean, "global": global_mean}

def target_encode_transform(df, encodings):
    df = df.copy()
    df["Store_Mean_Sales"] = df["Store"].map(encodings["store"]).fillna(encodings["global"])
    df["Dept_Mean_Sales"] = df["Dept"].map(encodings["dept"]).fillna(encodings["global"])
    key = pd.Series(list(zip(df["Store"], df["Dept"])), index=df.index)
    df["StoreDept_Mean_Sales"] = key.map(encodings["storedept"]).fillna(df["Dept_Mean_Sales"])
    return df

def weights_from_holiday(is_holiday):
    is_holiday = np.asarray(is_holiday).astype(bool)
    return np.where(is_holiday, 5.0, 1.0)

class LightGBMPipeline(BaseEstimator, RegressorMixin):

    def __init__(self, num_leaves=31, learning_rate=0.05, n_estimators=300,
                 max_depth=-1, min_child_samples=20, random_state=42):
        self.num_leaves = num_leaves
        self.learning_rate = learning_rate
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.min_child_samples = min_child_samples
        self.random_state = random_state

    def _build_features(self, X, encodings):
        df = merge_static(add_calendar_features(X), STORES_DF, FEATURES_DF)
        df = target_encode_transform(df, encodings)
        for c in FINAL_CAT_COLS:
            df[c] = df[c].astype("category")
        return df[FINAL_FEATURE_COLS]

    def predict(self, X):
        Xf = self._build_features(X, self.encodings_)
        preds = self.model_.predict(Xf)
        return np.clip(preds, 0, None)

class XGBoostPipeline(BaseEstimator, RegressorMixin):

    def __init__(self, max_depth=6, learning_rate=0.05, n_estimators=300,
                 subsample=0.8, colsample_bytree=0.8, min_child_weight=1, random_state=42):
        self.max_depth = max_depth
        self.learning_rate = learning_rate
        self.n_estimators = n_estimators
        self.subsample = subsample
        self.colsample_bytree = colsample_bytree
        self.min_child_weight = min_child_weight
        self.random_state = random_state

    def _build_features(self, X, encodings):
        df = merge_static(add_calendar_features(X), STORES_DF, FEATURES_DF)
        df = target_encode_transform(df, encodings)
        for c in FINAL_CAT_COLS:
            df[c] = df[c].astype("category")
        return df[FINAL_FEATURE_COLS]

    def predict(self, X):
        Xf = self._build_features(X, self.encodings_)
        preds = self.model_.predict(Xf)
        return np.clip(preds, 0, None)

class RowPredictAdapter:

    def __init__(self, pipe):
        self.pipe = pipe

    def predict(self, raw_df):
        df = raw_df.copy()
        df["Date"] = pd.to_datetime(df["Date"])
        preds = self.pipe.predict(df)
        return pd.DataFrame({
            "unique_id": df["Store"].astype(str) + "_" + df["Dept"].astype(str),
            "ds": df["Date"],
            "Weekly_Sales_Pred": np.asarray(preds, dtype=float),
        })

def load_gbm(pipeline_dir):
    global STORES_DF, FEATURES_DF, FINAL_FEATURE_COLS, FINAL_CAT_COLS
    STORES_DF = stores.copy()
    FEATURES_DF = features.copy()
    pkl = glob.glob(f"{pipeline_dir}/**/pipeline.pkl", recursive=True)[0]
    with open(pkl, "rb") as f:
        pipe = pickle.load(f)
    FINAL_FEATURE_COLS = list(pipe.feature_names_)
    FINAL_CAT_COLS = [c for c in ("Store", "Dept", "Type") if c in FINAL_FEATURE_COLS]
    print(type(pipe).__name__, "| n_features:", len(FINAL_FEATURE_COLS), "| cat:", FINAL_CAT_COLS)
    return RowPredictAdapter(pipe)

In [ ]:
SUPER_BOWL = pd.to_datetime(["2010-02-12", "2011-02-11", "2012-02-10", "2013-02-08"])
LABOR_DAY = pd.to_datetime(["2010-09-10", "2011-09-09", "2012-09-07", "2013-09-06"])
THANKSGIVING = pd.to_datetime(["2010-11-26", "2011-11-25", "2012-11-23", "2013-11-29"])
CHRISTMAS = pd.to_datetime(["2010-12-31", "2011-12-30", "2012-12-28", "2013-12-27"])

def build_holidays_df():
    frames = []
    for name, dates in [("SuperBowl", SUPER_BOWL), ("LaborDay", LABOR_DAY),
                         ("Thanksgiving", THANKSGIVING), ("Christmas", CHRISTMAS)]:
        frames.append(pd.DataFrame({"holiday": name, "ds": dates, "lower_window": 0, "upper_window": 0}))
    return pd.concat(frames, ignore_index=True)

HOLIDAYS_DF = build_holidays_df()

def cold_start_fallback_table(df, value_col="Weekly_Sales"):
    fallback_per_key = df.groupby(["Store", "Dept"])[value_col].apply(lambda s: s.tail(8).mean())
    dept_mean = df.groupby("Dept")[value_col].mean()
    global_mean = float(df[value_col].mean())
    return fallback_per_key, dept_mean, global_mean

class PerSeriesProphetForecaster(BaseEstimator, RegressorMixin):

    def __init__(self, changepoint_prior_scale=0.05, seasonality_prior_scale=10.0,
                 seasonality_mode="additive", holidays_prior_scale=10.0, min_obs=20):
        self.changepoint_prior_scale = changepoint_prior_scale
        self.seasonality_prior_scale = seasonality_prior_scale
        self.seasonality_mode = seasonality_mode
        self.holidays_prior_scale = holidays_prior_scale
        self.min_obs = min_obs

    def _fallback(self, key, dept):
        if key in self.fallback_value_:
            return self.fallback_value_[key]
        if dept in self.dept_mean_.index:
            return float(self.dept_mean_.loc[dept])
        return self.global_mean_

    def predict(self, X):
        preds = np.empty(len(X), dtype=float)
        X = X.reset_index(drop=True)
        for (store, dept), idx in X.groupby(["Store", "Dept"]).groups.items():
            key = (store, dept)
            rows = X.loc[idx]
            if key not in self.models_ or self.models_[key] is None:
                preds[idx] = self._fallback(key, dept)
                continue
            future = pd.DataFrame({"ds": pd.to_datetime(rows["Date"])})
            try:
                fc = self.models_[key].predict(future)
                preds[idx] = fc["yhat"].to_numpy()
            except Exception:
                preds[idx] = self._fallback(key, dept)
        return np.clip(preds, 0, None)

def load_prophet(pipeline_dir):
    import logging
    logging.getLogger("cmdstanpy").disabled = True
    from prophet import Prophet
    pkl = glob.glob(f"{pipeline_dir}/**/pipeline.pkl", recursive=True)[0]
    with open(pkl, "rb") as f:
        pipe = pickle.load(f)
    print("Prophet series models:", sum(1 for m in pipe.models_.values() if m is not None),
          "/", len(pipe.models_))
    return RowPredictAdapter(pipe)

## 4. გამარჯვებულის ჩამოტვირთვა და pipeline-ის აღდგენა

In [ ]:
LOADERS = {
    "ARIMA": load_lookup, "SARIMA": load_lookup,
    "NBEATS": load_nf, "DLinear": load_nf,
    "TFT": load_tft,
    "LightGBM": load_gbm, "XGBoost": load_gbm,
    "Prophet": load_prophet,
}

run = wandb.init(project=WANDB_PROJECT, group=BEST_MODEL, job_type="inference",
                 name=f"{BEST_MODEL}_Inference",
                 config={"selected_model": BEST_MODEL, "selected_wmae": BEST_WMAE,
                         "selection_metric": leaderboard.iloc[0]["metric"],
                         "best_config": BEST_CONFIG})
wandb.log({"leaderboard": wandb.Table(dataframe=leaderboard.drop(columns=["best_config"]).assign(
    best_config=leaderboard["best_config"].astype(str)))})

if BEST_MODEL in REGISTRY_MODELS:
    artifact_ref = f"wandb-registry-model/{BEST_MODEL}:latest"
else:
    artifact_ref = f"{BEST_MODEL}_pipeline:latest"
artifact = run.use_artifact(artifact_ref, type="model")
PIPELINE_DIR = artifact.download()
print(f"artifact '{artifact_ref}' ჩამოტვირთულია:", PIPELINE_DIR)

pipeline = LOADERS[BEST_MODEL](PIPELINE_DIR)
print("pipeline მზადაა:", type(pipeline).__name__, f"({BEST_MODEL})")

## 5. პროგნოზი raw `test.csv`-ზე

In [ ]:
preds = pipeline.predict(test_raw)
assert not preds["Weekly_Sales_Pred"].isna().any(), "პროგნოზში NaN დარჩა!"
print(preds.shape)
preds.head()

## 6. Kaggle submission ფორმატში დაკონვერტირება

Kaggle-ს სჭირდება ორსვეტიანი CSV, `Id` (`{Store}_{Dept}_{Date}`) და
`Weekly_Sales`, ზუსტად `sampleSubmission.csv`-ის row-order-ით.

In [ ]:
submission = preds.copy()
submission["Id"] = submission["unique_id"] + "_" + submission["ds"].dt.strftime("%Y-%m-%d")
submission = submission.rename(columns={"Weekly_Sales_Pred": "Weekly_Sales"})[["Id", "Weekly_Sales"]]
submission = submission.drop_duplicates(subset="Id")

sample_sub = pd.read_csv(glob.glob(f"{DATA_DIR}/sampleSubmission.csv*")[0])
missing_ids = set(sample_sub["Id"]) - set(submission["Id"])
assert not missing_ids, f"{len(missing_ids)} Id აკლია submission-ს, მაგ: {list(missing_ids)[:5]}"
submission = submission.set_index("Id").loc[sample_sub["Id"]].reset_index()

SUBMISSION_PATH = f"./submission_{BEST_MODEL.lower()}.csv"
submission.to_csv(SUBMISSION_PATH, index=False)
print("submission შენახულია:", SUBMISSION_PATH, "| rows:", len(submission))
submission.head()

## 7. Submission-ის დალოგვა Wandb-ზე

submission ინახება artifact-ად გამარჯვებული მოდელის ჯგუფში — ტრასირებადია,
რომელი მოდელის რომელი pipeline-ვერსიით დაგენერირდა.

In [ ]:
sub_artifact = wandb.Artifact(name=f"{BEST_MODEL}_submission", type="submission",
                               metadata={"pipeline_artifact": artifact.name,
                                         "selected_model": BEST_MODEL,
                                         "selection_wmae": BEST_WMAE,
                                         "n_rows": len(submission)})
sub_artifact.add_file(SUBMISSION_PATH)
run.log_artifact(sub_artifact)
wandb.log({"n_rows": len(submission)})
run.finish()
print(f"{BEST_MODEL} submission artifact დალოგილია.")